### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### Read all the RACV pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: era-terms-and-conditions-2023-24.pdf
  ✓ Loaded 16 pages

Processing: motor-insurance-complete-care-pds-current.pdf
  ✓ Loaded 53 pages

Total documents loaded: 69


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, page_content='Contact Us \n13 72 28\nracv.com.au\nOr visit an RACV Store\nROYAL AUTOMOBILE CLUB  \nOF VICTORIA (RACV) LTD  \nABN 44 044 060 833 \nLevel 7, 485 Bourke Street  \nMelbourne VIC 3000\nEmergency \nRoadside\nAssistance\nTerms &\nConditions\n616674  01/23'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'sourc

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 69 documents into 200 chunks

Example chunk:
Content: Contact Us 
13 72 28
racv.com.au
Or visit an RACV Store
ROYAL AUTOMOBILE CLUB  
OF VICTORIA (RACV) LTD  
ABN 44 044 060 833 
Level 7, 485 Bourke Street  
Melbourne VIC 3000
Emergency 
Roadside
Assista...
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, page_content='Contact Us \n13 72 28\nracv.com.au\nOr visit an RACV Store\nROYAL AUTOMOBILE CLUB  \nOF VICTORIA (RACV) LTD  \nABN 44 044 060 833 \nLevel 7, 485 Bourke Street  \nMelbourne VIC 3000\nEmergency \nRoadside\nAssistance\nTerms &\nConditions\n616674  01/23'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'sourc

### Embedding and VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings"""

        self.model_name =  model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model =  SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts

            Args:
                texts: List of text strings to embed
            return:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initilize the Embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2151.78it/s]


Model loaded successfully. Embedding dimesions: 384


C:\Users\kirth\AppData\Local\Temp\ipykernel_15868\60672882.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ Initialize the vectore store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, page_content='Contact Us \n13 72 28\nracv.com.au\nOr visit an RACV Store\nROYAL AUTOMOBILE CLUB  \nOF VICTORIA (RACV) LTD  \nABN 44 044 060 833 \nLevel 7, 485 Bourke Street  \nMelbourne VIC 3000\nEmergency \nRoadside\nAssistance\nTerms &\nConditions\n616674  01/23'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': '2023-01-13T13:01:09+11:00', 'moddate': '2023-01-13T13:01:16+11:00', 'trapped': '/False', 'source': '..\\data\\pdfs\\era-terms-and-conditions-2023-24.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'sourc

In [11]:
texts=[doc.page_content for doc in chunks]
texts

['Contact Us \n13 72 28\nracv.com.au\nOr visit an RACV Store\nROYAL AUTOMOBILE CLUB  \nOF VICTORIA (RACV) LTD  \nABN 44 044 060 833 \nLevel 7, 485 Bourke Street  \nMelbourne VIC 3000\nEmergency \nRoadside\nAssistance\nTerms &\nConditions\n616674  01/23',
 'Introduction 4\nRACV Emergency Roadside Assistance 6\nObtaining Assistance 6\nProof of ERA product holding 6\nERA Products Features Table 7\n1. RACV Total Care 8\n 1.1 Emergency Roadside Assistance 8\n 1.2\t Extended\tBenefits\t 8\n\t \t General\tBenefits\t 8\n  Benefits\twhen\tunder\t100km\tfrom\thome\taddress 9\n\t \t Benefits\twhen\tover\t100km\tfrom\thome\taddress\t 9\n\t 1.3\t Personal\tBenefits\t 10\n2.\t RACV\tExtra\tCare\t 10\n\t 2.1\t Emergency\tRoadside\tAssistance\t 10\n\t 2.2\t Extended\tBenefits\t 11\n\t \t General\tBenefits\t 11\n  Benefits\twhen\tunder\t100km\tfrom\thome\taddress 11\n\t \t Benefits\twhen\tover\t100km\tfrom\thome\taddress\t 11\n3.  RACV Roadside Care 12\n 3.1 Emergency Roadside Assistance 12\n\t 3.2\t E

In [12]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 200 texts...


Batches: 100%|██████████| 7/7 [00:12<00:00,  1.82s/it]


Generated embeddings with shape: (200, 384)
Adding 200 documents to vector store...
Successfully added 200 documents to vector store
Total documents in collection: 200


### Retriever Pipeline From VectorStore

In [77]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float= 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold
            Returns:
                List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for the query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results ['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} docs (after filtering)")
                return retrieved_docs
                
            else:
                print(f"Error during retrieval: {e}")
                return []
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)                   

In [75]:
rag_retriever

In [78]:
rag_retriever.retrieve("Does RACV cover for accidental damage to your vehicle in Complete care?")

Retrieving documents for the query: Does RACV cover for accidental damage to your vehicle in Complete care?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.31it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 docs (after filtering)


[{'id': 'doc_c9f7cdc6_38',
  'content': 'provided\tfor\tin\tthese\tTerms\tand\tConditions.\t\n •\t \tWhere\ta\tVehicle\tis\tleaking\tgas\tor\tfuel,\tRACV\tmay\trefuse\t\nto\tattend\tthe\tvehicle\tunless\tand\tuntil\temergency\t\nservices\thave\tattended\tand\tdeemed\tthe\tVehicle\tand\t\nenvironment\tsafe.\t\n •\t \tRACV\tmay\taccept\tliability\tfor\tdamage\tto\ta\tVehicle\t\ndirectly\tcaused\tby\tService\tto\tit\twhere\tnotice\tof\tthat\t\ndamage\tis\tgiven\tto\tRACV\twithin\t7\tdays\tof\tservice,\tRACV\t\nis\tgiven\tan\topportunity\tto\tinspect\tthe\tvehicle\tbefore\t\nrepairs\tare\tcarried\tout\tand\tconsiders\tthat\tthe\tdamage\t\nwas\tdirectly\tcaused\tby\tnegligence\ton\tthe\tpart\tof\t\nRACV\tin\tproviding\tthe\tservice.\tRACV\tis\tunder\tno\tstrict\t\nobligation\tto\taccept\tliability.\t\n •\t \tNotwithstanding\tthe\tprevious\tparagraph,\tRACV\tdoes\tnot\t\nrepresent\tthat\tany\tVehicle\t(or\tany\tpart\tthereof)\tto\twhich\t\nit\tprovides\tService\twill\tbe,\tor\twill\tremain\t

In [87]:
rag_retriever.retrieve("How many litres of petrol does RACV provide in an emergency?")


Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.74it/s]

Generated embeddings with shape: (1, 384)


Retrieved 5 docs (after filtering)


[{'id': 'doc_a08338c2_18',
  'content': 'Caravan\tis\tbeing\ttowed\tby\tan\tEligible\tVehicle.\nTravel Party Recovery Transport Benefits – \nillness, injury or hospitalisation\nIf\tyou\tor\tyour\tTravel\tParty\tsuffer\ta\tmedical\temergency\t\nrequiring\thospitalisation\tfor\t7\tdays\tor\tmore,\tor\t\nexperience\tan\tillness\tor\tinjury\tleaving\tyou\tunfit\tto\tdrive,\t\nRACV\tmay\tprovide\tup\tto\t$1,000\tin\ttotal\tper\tincident\tfor\t\naccommodation,\tvehicle\ttransport\tfor\tthe\nMember\tand\tpassenger\ttransport\tfor\tthe\tMember\tand\t\nTravel\tParty\tto\tjoin\tthe\tMember\tor\treturn\thome\t(subject\tto\t\nthe\ttreating\tdoctor’s\tconsent,\tin\twriting\tas\trequired\tby\tRACV).\t\n8 9',
  'metadata': {'source_file': 'era-terms-and-conditions-2023-24.pdf',
   'trapped': '/False',
   'creationdate': '2023-01-13T13:01:09+11:00',
   'creator': 'Adobe InDesign 18.1 (Macintosh)',
   'doc_index': 18,
   'total_pages': 16,
   'page_label': '5',
   'page': 4,
   'content_length': 570,
 

In [79]:
rag_retriever.retrieve("Tell me about Claiming under my Policy ?")


Retrieving documents for the query: Tell me about Claiming under my Policy ?
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 docs (after filtering)


[{'id': 'doc_f4640fd8_133',
  'content': 'Claiming under your Policy \nWe are available to help you 24 hours a day, 7 days a week on 1300 654 947. \nIf you make a claim, we will: \nask a series of questions, or ask for detailed written information \ngive immediate assistance with the claim \ntell you if you need to pay any excess and how to pay it. \n20',
  'metadata': {'total_pages': 53,
   'source': '..\\data\\pdfs\\motor-insurance-complete-care-pds-current.pdf',
   'file_type': 'pdf',
   'page': 26,
   'content_length': 307,
   'moddate': '2023-08-21T11:27:04+10:00',
   'creationdate': '2022-05-27T09:52:32+10:00',
   'producer': 'PyPDF',
   'source_file': 'motor-insurance-complete-care-pds-current.pdf',
   'creator': 'PyPDF',
   'doc_index': 133,
   'page_label': '27',
   'subject': 'RACV Complete Care Motor Insurance SPDS'},
  'similarity_score': 0.2531358003616333,
  'distance': 0.7468641996383667,
  'rank': 1},
 {'id': 'doc_d84d0114_134',
  'content': 'Your responsibilities \nwhe

### Integration Vectordb Context pipeline With LLM output

In [82]:
### Simple RAG pipeline with Groq LLM
from langchain_anthropic import ChatAnthropic
import os
from dotenv import load_dotenv
load_dotenv()

claude_api_key = os.getenv("CLAUDE_API_KEY")

llm = ChatAnthropic(anthropic_api_key=claude_api_key, model="claude-sonnet-4-5-20250929", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)

    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [88]:
answer = rag_simple("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm)
print(answer)

Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 47.01it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 docs (after filtering)


Based on the provided context, there is no information about how many litres of petrol RACV provides in an emergency. The context does not mention emergency petrol provision or fuel delivery services.


### Enhanced RAG Pipeline Features

In [86]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.10it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 docs (after filtering)
Answer: Based on the provided context, there is no information about RACV providing petrol/fuel in an emergency. The context only mentions coverage for accommodation, vehicle transport, and passenger transport in cases of medical emergencies requiring hospitalization or illness/injury that leaves someone unfit to drive.
Sources: [{'source': 'era-terms-and-conditions-2023-24.pdf', 'page': 4, 'score': 0.11056196689605713, 'preview': 'Caravan\tis\tbeing\ttowed\tby\tan\tEligible\tVehicle.\nTravel Party Recovery Transport Benefits – \nillness, injury or hospitalisation\nIf\tyou\tor\tyour\tTravel\tParty\tsuffer\ta\tmedical\temergency\t\nrequiring\thospitalisation\tfor\t7\tdays\tor\tmore,\tor\t\nexperience\tan\tillness\tor\tinjury\tleaving\tyou\tunfit\tto\tdrive,\t\nRACV\tma...'}]
Confidence: 0.11056196689605713
Context Preview: Caravan	is	being	towed	by	an	Eligible	Vehicle.
Travel Party Recovery Transport Benefits –

In [89]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("How many litres of petrol does RACV provide in an emergency?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for the query: How many litres of petrol does RACV provide in an emergency?
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 docs (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Caravan	is	being	towed	by	an	Eligible	Vehicle.
Travel Party Recovery Transport Benefits – 
illness, injury or hospitalisation
If	you	or	your	Travel	Party	suffer	a	medical	emergency	
requiring	hospitalisation	for	7	days	or	more,	or	
experience	an	illness	or	injury	leaving	you	unfit	to	drive,	
RACV	may	provide	up	to	$1,000	in	total	per	incident	for	
accommodation,	vehicle	transport	for	the
Member	and	passenger	transport	for	the	Member	and	
Travel	Party	to	join	the	Member	or	return	home	(subject	to	
the	treating	doctor’s	consent,	in	writing	as	required	by	RACV).	
8 9

Question: How many litres of petrol does RACV provide in an emergency?

Answer:

Final Answer: Based on the provided context, there is no information about RACV providing petrol/fuel in an emergency. The context only mentions coverage for accommodation, vehicle tr